# 07: Model Comparison & Visualizations

**Goal:** Compare all trained models and generate final visualizations for the report.

## Outputs
- `results/tables/model_comparison.csv` - Performance table
- `results/figures/roc_curves.png` - ROC curves overlay
- `results/figures/confusion_matrices.png` - Confusion matrices grid

In [ ]:
import sys
sys.path.insert(0, "../")

import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, roc_curve, auc
from src import data_utils, nlp_utils, model_utils, viz_utils

SEED = 42
np.random.seed(SEED)
import random
random.seed(SEED)

print("Imports successful!")

## Step 1: Load All Results

In [ ]:
# Load model CV scores from previous notebooks
logistic_results = pd.read_csv("../results/tables/logistic_cv_scores.csv")
lasso_ridge_results = pd.read_csv("../results/tables/lasso_ridge_comparison.csv")
rf_results = pd.read_csv("../results/tables/rf_cv_scores.csv")

# Combine into one comparison table
all_results = pd.concat([
    logistic_results,
    lasso_ridge_results,
    rf_results
], ignore_index=True)

print("\n=== Model Comparison ===")
print(all_results[['model', 'accuracy_mean', 'auc_roc_mean', 'f1_mean', 'precision_mean', 'recall_mean']])

In [ ]:
# Try to load BERT results if they exist (optional)
try:
    bert_results = pd.read_csv("../results/tables/bert_scores.csv")
    bert_results['model'] = 'BERT'
    # Reformat columns if needed
    if 'eval_accuracy' in bert_results.columns:
        bert_results['accuracy_mean'] = bert_results['eval_accuracy']
    all_results = pd.concat([all_results, bert_results], ignore_index=True)
    print("\n✓ Included BERT results")
except FileNotFoundError:
    print("\n(BERT results not found - optional notebook)")

In [ ]:
# Save final comparison table
all_results.to_csv("../results/tables/model_comparison.csv", index=False)
print("\nSaved to results/tables/model_comparison.csv")

## Step 2: Prepare Data for Visualizations

In [ ]:
# Load data and features
df = pd.read_csv("../data/processed/bills_speeches_preprocessed.csv")
X_text = df['speeches_combined'].values
y = df['passed'].values

# TF-IDF
X_tfidf, vectorizer, feature_names = nlp_utils.create_tfidf_features(
    X_text, max_features=5000, min_df=5, max_df=0.95
)
X_dense = X_tfidf.toarray()

print(f"Data loaded: {len(df)} bills")

## Step 3: Generate ROC Curves

In [ ]:
# Train models for ROC predictions
from sklearn.model_selection import cross_val_predict

# Logistic Regression
lr_model = model_utils.train_logistic_regression(X_tfidf, y, random_state=SEED)
lr_proba = cross_val_predict(
    lr_model, X_tfidf, y, cv=5, method="predict_proba"
)[:, 1]

# LASSO
lasso_model = model_utils.train_lasso_logistic(X_tfidf, y, cv_splits=5, random_state=SEED)
lasso_proba = cross_val_predict(
    lasso_model, X_tfidf, y, cv=5, method="predict_proba"
)[:, 1]

# Ridge
ridge_model = model_utils.train_ridge_logistic(X_tfidf, y, cv_splits=5, random_state=SEED)
ridge_proba = cross_val_predict(
    ridge_model, X_tfidf, y, cv=5, method="predict_proba"
)[:, 1]

# Random Forest
rf_model = model_utils.train_random_forest(X_dense, y, n_estimators=200, random_state=SEED)
rf_proba = cross_val_predict(
    rf_model, X_dense, y, cv=5, method="predict_proba"
)[:, 1]

print("✓ Models trained for ROC curves")

In [ ]:
# Plot ROC curves
models_data = [
    ('Logistic Regression', y, lr_proba),
    ('LASSO', y, lasso_proba),
    ('Ridge', y, ridge_proba),
    ('Random Forest', y, rf_proba),
]

viz_utils.plot_roc_curves(
    models_data,
    output_path="../results/figures/roc_curves.png"
)

## Step 4: Generate Confusion Matrices

In [ ]:
# Get predictions for confusion matrices
lr_pred = cross_val_predict(lr_model, X_tfidf, y, cv=5)
lasso_pred = cross_val_predict(lasso_model, X_tfidf, y, cv=5)
ridge_pred = cross_val_predict(ridge_model, X_tfidf, y, cv=5)
rf_pred = cross_val_predict(rf_model, X_dense, y, cv=5)

# Get confusion matrices
models_cm = [
    ('Logistic Regression', confusion_matrix(y, lr_pred)),
    ('LASSO', confusion_matrix(y, lasso_pred)),
    ('Ridge', confusion_matrix(y, ridge_pred)),
    ('Random Forest', confusion_matrix(y, rf_pred)),
]

# Plot
viz_utils.plot_confusion_matrices(
    models_cm,
    output_path="../results/figures/confusion_matrices.png"
)

## Step 5: Summary Statistics for Report

In [ ]:
# Print summary for report writing
print("\n" + "="*60)
print("FINAL MODEL COMPARISON")
print("="*60)

summary_cols = ['model', 'accuracy_mean', 'auc_roc_mean', 'f1_mean']
summary_table = all_results[summary_cols].sort_values('auc_roc_mean', ascending=False)

print("\n" + summary_table.to_string(index=False))

# Best model
best_model = summary_table.iloc[0]
print(f"\n✓ Best model: {best_model['model']} (AUC-ROC: {best_model['auc_roc_mean']:.4f})")

## Step 6: Key Insights for Report

In [ ]:
# Load feature lists for interpretation
lasso_features = pd.read_csv("../results/tables/lasso_top_features.csv")
rf_features = pd.read_csv("../results/tables/rf_top_features.csv")

print("\n" + "="*60)
print("KEY FINDINGS FOR REPORT")
print("="*60)

print("\n1. LASSO FEATURES (Words predicting passage/failure):")
print("\n   Positive (predict PASSAGE):")
positive = lasso_features[lasso_features['coefficient'] > 0].head(5)
for _, row in positive.iterrows():
    print(f"     - {row['feature']}: {row['coefficient']:.4f}")

print("\n   Negative (predict FAILURE):")
negative = lasso_features[lasso_features['coefficient'] < 0].head(5)
for _, row in negative.iterrows():
    print(f"     - {row['feature']}: {row['coefficient']:.4f}")

print("\n2. RANDOM FOREST TOP-5 FEATURES:")
for _, row in rf_features.head(5).iterrows():
    print(f"   - {row['feature']}: {row['importance']:.4f}")

In [ ]:
print("\n" + "="*60)
print("NEXT: Write final_report.md in report/ folder")
print("="*60)
print("\nReport structure:")
print("  1. Research Question (1 paragraph)")
print("  2. Data & Methods (1 page)")
print("  3. Results (1 page)")
print("  4. Discussion & Interpretation (1-2 pages)")
print("  5. Limitations (1 paragraph)")
print("\nKey interpretation questions:")
print("  - Which words predict passage? Why?")
print("  - Which model performs best and why?")
print("  - What is the baseline model (null result)?")
print("  - What confounders might explain the results?")